In [1]:
# Import libraries 
import multiconductor as mc
import numpy as np
from calculate_impedance_matrix import calc_three_phase

In [2]:
mc_net = mc.create_empty_network()
mc.create_bus(mc_net, vn_kv=0.4, num_phases=4, name="Bus 0", grounded_phases=None)  
mc.create_bus(mc_net, vn_kv=0.4, num_phases=4, name="Bus 1", grounded_phases=None)

np.int64(1)

In [3]:
# Create wire specified in IEEE 13 node test feeder Config 601 with spacing ID 500 for three-phase 4 wire
# Config.   Phasing     Phase           Neutral     Spacing ID
# 601	    B A C N	    556,500 26/7	4/0 6/1	    500
# A: 
# B: 
# C: 
# N: 

INCH = 0.0254  # in meters
FEET = 0.3048  # in meters
MILE = 1609.34 # in meters

wires = [
    {"location": (0.0, 28 * FEET), "r_ac": 0.1859 / MILE , "gmr": 0.0313 * FEET},
    {"location": (-2.5 * FEET, 28 * FEET), "r_ac": 0.1859 / MILE, "gmr": 0.0313 * FEET},
    {"location": (4.5 * FEET, 28 * FEET), "r_ac": 0.1859 / MILE, "gmr": 0.0313 * FEET},
    {"location": (0.5 * FEET, 24 * FEET), "r_ac": 0.592 / MILE, "gmr": 0.00814 * FEET},
]

z_M = calc_three_phase(wires=wires)





In [4]:
np.set_printoptions(precision=6, suppress=True)

print(np.array2string(
    z_M,
    formatter={"complex_kind": lambda value: f"{value.real:.6f} {value.imag:+.6f}j"},
))

[[0.216526 +0.628585j 0.098756 +0.305639j 0.097462 +0.265521j]
 [0.098756 +0.305639j 0.212114 +0.643016j 0.095366 +0.239177j]
 [0.097462 +0.265521j 0.095366 +0.239177j 0.209677 +0.651079j]]


In [ ]:
# Note that the impedance matrix returned above is already in ohm per km and kron reduced to 3 by 3 (neutral eliminated)

num_conductors = 3

rmatrix = np.real(z_M)
xmatrix = np.imag(z_M)

mdata = {   
            "r_1_ohm_per_km": rmatrix[:, 0],
            "x_1_ohm_per_km": xmatrix[:, 0],
            "g_1_us_per_km": np.zeros(num_conductors),
            "b_1_us_per_km": np.zeros(num_conductors),
            "max_i_ka": np.ones(num_conductors),
            "r_2_ohm_per_km": rmatrix[:, 1],
            "x_2_ohm_per_km": xmatrix[:, 1],
            "g_2_us_per_km": np.zeros(num_conductors),
            "b_2_us_per_km": np.zeros(num_conductors),
            "r_3_ohm_per_km": rmatrix[:, 2],
            "x_3_ohm_per_km": xmatrix[:, 2],
            "g_3_us_per_km": np.zeros(num_conductors),
            "b_3_us_per_km": np.zeros(num_conductors),
        } 


mc.pycci.std_types.create_std_types(mc_net, {"config_601": mdata}, element="matrix")

mc.create_line(mc_net, model_type="matrix", std_type="config_601", from_bus=1, from_phase=(1, 2, 3),
                   to_bus=2, to_phase=(1, 2, 3), length_km=0.5, name="Line_config_601") 

np.int64(0)